# 05 — SCIN Dataset: Gaussian Noise Augmentation

Downloads SCIN images, creates Gaussian-noised copies, and uploads
original + noised images to our GCS bucket (`skin-tone-project`).

**What this does:**
1. Download SCIN images from public bucket
2. Count images BEFORE
3. Create a noised copy of every image (Gaussian noise, std=25)
4. Count images AFTER
5. Upload everything to our bucket

**References:**
- SCIN dataset: https://github.com/google-research-datasets/scin
- Augmentation: https://pmc.ncbi.nlm.nih.gov/articles/PMC5977656/

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
sys.path.insert(0, "/content/NST_Class")

print("Setup complete!")

In [ ]:
# Cell 2: Download SCIN images from public bucket
!mkdir -p data/scin/images
!gsutil -m cp -r -n gs://dx-scin-public-data/dataset/images/ data/scin/images/
# -m = parallel, -n = skip existing

In [ ]:
# Cell 3: Count images BEFORE augmentation
import glob

original_images = [f for f in glob.glob("data/scin/images/**/*", recursive=True) if os.path.isfile(f)]
count_before = len(original_images)

print("=" * 50)
print(f"BEFORE AUGMENTATION: {count_before} images")
print("=" * 50)

# Show a few example filenames
for f in original_images[:5]:
    print(f"  {os.path.basename(f)}")
if count_before > 5:
    print(f"  ... and {count_before - 5} more")

In [ ]:
# Cell 4: Add Gaussian noise to every image
#
# From PMC5977656: "We generate an array N where each element is a sample
# from a gaussian distribution with mu=0 and sigma in range [0.1, 0.9]."
#
# std=25 on [0,255] scale — moderate noise that adds variation
# without destroying the image.

import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

INPUT_DIR = "data/scin/images"
NOISE_STD = 25.0
NOISE_MEAN = 0.0
SEED = 42

np.random.seed(SEED)

extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_files = [
    f for f in Path(INPUT_DIR).rglob("*")
    if f.is_file()
    and f.suffix.lower() in extensions
    and "_noised" not in f.stem  # don't noise already-noised images
]

print(f"Images to noise: {len(image_files)}")

created = 0
failed = 0

for img_path in tqdm(image_files, desc="Adding Gaussian noise"):
    try:
        img = Image.open(img_path).convert("RGB")
        arr = np.array(img)

        # Add Gaussian noise: I' = I + N(mean, std)
        noise = np.random.normal(NOISE_MEAN, NOISE_STD, arr.shape)
        noised = np.clip(arr.astype(np.float64) + noise, 0, 255).astype(np.uint8)
        noised_img = Image.fromarray(noised)

        # Save next to original: image123.jpg -> image123_noised.jpg
        noised_path = img_path.parent / f"{img_path.stem}_noised{img_path.suffix}"
        noised_img.save(noised_path)
        created += 1
    except Exception as e:
        failed += 1
        print(f"Failed: {img_path.name}: {e}")

print(f"\nCreated: {created} noised images")
if failed:
    print(f"Failed:  {failed}")

In [ ]:
# Cell 5: Count images AFTER augmentation
all_images = [f for f in glob.glob("data/scin/images/**/*", recursive=True) if os.path.isfile(f)]
noised_images = [f for f in all_images if "_noised" in os.path.basename(f)]
count_after = len(all_images)

print("=" * 50)
print("AUGMENTATION SUMMARY")
print("=" * 50)
print(f"BEFORE:    {count_before} images (original)")
print(f"CREATED:   {len(noised_images)} noised copies")
print(f"AFTER:     {count_after} images total")
print(f"")
print(f"Original:  {count_after - len(noised_images)}")
print(f"Noised:    {len(noised_images)}")
print(f"Ratio:     {count_after / count_before:.1f}x the original dataset")

In [ ]:
# Cell 6: Upload everything to your GCS bucket
!gsutil -m cp -r data/scin/images/ gs://skin-tone-project/scin_images/
print("\nUploaded to gs://skin-tone-project/scin_images/")

In [ ]:
# Cell 7: Verify bucket contents
!echo "Total files in bucket:"
!gsutil ls gs://skin-tone-project/scin_images/** | wc -l
!echo "\nSample files:"
!gsutil ls gs://skin-tone-project/scin_images/** | head -10